In [1]:
import cobra

In [ ]:
model = cobra.io.read_sbml_model("../../../model.xml")

In [3]:
model.metabolites.cpd02711_c0

Metabolite identifier,cpd02711_c0
Name,2-Keto-3-deoxy-6-phosphogluconate
Memory address,0x119599610
Formula,C6H8O9P
Compartment,c0
In 3 reaction(s),"rxn03884_c0, rxn01477_c0, rxn01123_c0"


In [4]:
# Make an external version of the KDPG metabolite
kdpg_e0_met = cobra.Metabolite("cpd02711_e0",
    compartment="e0",
    name=model.metabolites.cpd02711_c0.name,
    formula=model.metabolites.cpd02711_c0.formula,
    charge=model.metabolites.cpd02711_c0.charge)
kdpg_e0_met

Metabolite identifier,cpd02711_e0
Name,2-Keto-3-deoxy-6-phosphogluconate
Memory address,0x11b3913d0
Formula,C6H8O9P
Compartment,e0
In 0 reaction(s),


In [5]:
# Add a transport reaction for KDPG
kdpg_trans = cobra.Reaction("kdpg_transport")
kdpg_trans.name = "KDPG transport"
kdpg_trans.lower_bound = -1000  # Allow uptake
kdpg_trans.upper_bound = 1000   # Allow secretion
kdpg_trans.add_metabolites({
    model.metabolites.cpd02711_c0: -1,  # KDPG in the cytosol
    kdpg_e0_met: 1,   # KDPG in the extracellular space
})
model.add_reactions([kdpg_trans])
model.reactions.kdpg_transport

Reaction identifier,kdpg_transport
Name,KDPG transport
Memory address,0x11b3a4a90
Stoichiometry,cpd02711_c0 <=> cpd02711_e0 2-Keto-3-deoxy-6-phosphogluconate <=> 2-Keto-3-deoxy-6-phosphogluconate
GPR,
Lower bound,-1000
Upper bound,1000


In [6]:
# Add an exchange reaction for the external KDPG
model.add_boundary(model.metabolites.cpd02711_e0, type="exchange")
model.reactions.EX_cpd02711_e0

Reaction identifier,EX_cpd02711_e0
Name,2-Keto-3-deoxy-6-phosphogluconate exchange
Memory address,0x11b3b2790
Stoichiometry,cpd02711_e0 <=> 2-Keto-3-deoxy-6-phosphogluconate <=>
GPR,
Lower bound,-1000.0
Upper bound,1000.0


In [7]:
# Define a medium
medium = {
    "EX_cpd02711_e0": 10,  # KDPG
    "EX_cpd00007_e0": 20,  # Oxygen
    "EX_cpd00013_e0": 1000,  # Ammonia
    "EX_cpd00011_e0": 1000,  # CO2
    "EX_cpd00067_e0": 1000,  # H+
    "EX_cpd00009_e0": 1000,  # Phosphate
    "EX_cpd00001_e0": 1000,  # H2O
    "EX_cpd00063_e0": 1000,  # Ca2+
    "EX_cpd00099_e0": 1000,  # Cl-
    "EX_cpd00149_e0": 1000,  # Co2+
    "EX_cpd00058_e0": 1000,  # Cu2+
    "EX_cpd00254_e0": 1000,  # Mg2+
    "EX_cpd00205_e0": 1000,  # K+
    "EX_cpd00971_e0": 1000,  # Na+
    "EX_cpd00048_e0": 1000,  # Sulfate
    "EX_cpd00034_e0": 1000,  # Zn2+
    "EX_cpd10516_e0": 1000,  # Fe+3
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
}
model.medium = medium

In [8]:
solution = model.optimize()
solution

,fluxes,reduced_costs
rxn02201_c0,0.004925,4.779346e-18
rxn00351_c0,0.000000,-1.584772e-02
rxn07431_c0,0.000000,-2.145305e-17
rxn00836_c0,0.000000,-7.923860e-03
rxn00423_c0,0.000000,-8.948932e-17
...,...,...
rxn01993_c0,0.016417,-1.760828e-18
rxn02788_c0,0.016417,0.000000e+00
rxn01894_c0,0.016417,5.008957e-18
kdpg_transport,-10.000000,7.836957e-18


In [9]:
# Save the map
cobra.io.write_sbml_model(model, "model_w_kdg_ex.xml")

In [ ]:
# Save the fluxes as a JSON file
solution.fluxes.to_json("kdg_fluxes.json")